In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

sns.set_theme(style="whitegrid")

In [4]:
df = pd.read_csv('train.csv').copy()
target_col = 'Loan_Status'  # Change to 'rating' or 'Class' depending on file

# Drop ID columns if they exist
if 'Loan_ID' in df.columns:
    df = df.drop('Loan_ID', axis=1)

print(f'Dataset Shape: {df.shape}')
print('\nFirst 5 rows:')
print(df.head())

print('\nMissing Values:')
print(df.isna().sum())

FileNotFoundError: [Errno 2] No such file or directory: 'train.csv'

In [ ]:
# Target Distribution Plot
plt.figure(figsize=(8,5))
sns.countplot(x=target_col, data=df, palette='magma')
plt.title("Distribution of Ratings (Target)")
plt.show()

In [ ]:
# 1. Fill missing values
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].fillna(df[col].mode()[0])
    else:
        df[col] = df[col].fillna(df[col].median())

# 2. Encode Categorical features
le = LabelEncoder()
for col in df.select_dtypes(include=['object']).columns:
    df[col] = le.fit_transform(df[col])

X = df.drop(target_col, axis=1)
y = df[target_col]

# 3. Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 4. PCA (Dimensionality Reduction)
pca = PCA(n_components=0.95)
X_pca = pca.fit_transform(X_scaled)

# 5. Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_pca, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Features reduced from {X.shape[1]} to {X_pca.shape[1]} via PCA.")

In [ ]:
# Define models and hyperparameters
models = [
    ('Logistic_Lasso', GridSearchCV(LogisticRegression(solver='liblinear'), 
                                   {'penalty': ['l1'], 'C': [0.1, 1, 10]}, cv=5)),
    
    ('KNN', GridSearchCV(KNeighborsClassifier(), 
                         {'n_neighbors': [3, 5, 7]}, cv=5)),
    
    ('SVM', GridSearchCV(SVC(), 
                         {'C': [0.1, 1, 10], 'kernel': ['rbf']}, cv=5)),
    
    ('DecisionTree', GridSearchCV(DecisionTreeClassifier(), 
                                  {'max_depth': [None, 5, 10], 'criterion': ['gini', 'entropy']}, cv=5)),
    
    ('RandomForest', GridSearchCV(RandomForestClassifier(), 
                                  {'n_estimators': [50, 100]}, cv=3))
]

results = []

# Loop through models
for name, grid in models:
    grid.fit(X_train, y_train)
    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test)

    results.append({
        'Model': name,
        'Best Params': grid.best_params_,
        'CV Best Score': round(grid.best_score_, 4),
        'Test Accuracy': round(accuracy_score(y_test, y_pred), 4)
    })

# Display final comparison table
results_df = pd.DataFrame(results).sort_values(by='Test Accuracy', ascending=False).reset_index(drop=True)
print("\nFinal Model Comparison:")
print(results_df)

print(f"\nRecommended Model: {results_df.loc[0, 'Model']}")